In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [5]:
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal
from langchain.messages import HumanMessage, AIMessage, SystemMessage

In [7]:
from typing import Annotated

from langgraph.graph import add_messages


llm = init_chat_model("google_genai:gemini-2.5-flash")

class State(TypedDict):
    messages: Annotated[list, add_messages] #Append the messages form the user

def chatbot(state: State) -> State:
    return {"messages": [llm.invoke(state["messages"])]}

builder = StateGraph(State)
builder.add_node("chatbot_node", chatbot)

builder.add_edge(START, "chatbot_node")
builder.add_edge("chatbot_node", END)

graph = builder.compile()

In [4]:
message = {"role": "user", "content": "Who walked on the moon for the first time? Name only"}

response = graph.invoke({"messages": [message]})

response["messages"]

[HumanMessage(content='Who walked on the moon for the first time? Name only', additional_kwargs={}, response_metadata={}, id='cbbaed35-167e-4cae-aa09-4e31912327ec'),
 AIMessage(content='Neil Armstrong', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d0f43-ff3f-7e40-a55d-33ce1f4f285c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 39, 'total_tokens': 52, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 37}})]

In [8]:
state = None
print("Chatbot started! Type 'quit' or 'exit' to stop.")
while True:
    in_message = input("You: ")
    if in_message.lower() in {"quit", "exit"}:
        break

    # Add user message
    if state is None:
        state = {"messages": [HumanMessage(content=in_message)]}
    else:
        state["messages"].append(HumanMessage(content=in_message))

    # Call LLM
    print("Bot is thinking...")
    state = graph.invoke(state)

    # Print latest AI response
    ai_msg = [m for m in state["messages"] if isinstance(m, AIMessage)][-1]
    print("Bot:", ai_msg.content)

Chatbot started! Type 'quit' or 'exit' to stop.
Bot is thinking...
Bot: Hello! How can I help you today?
Bot is thinking...


c:\Users\PC\AppData\Local\Programs\Python\Python313\Lib\site-packages\langchain_google_genai\chat_models.py:2858: UserWarning: HumanMessage with empty content was removed to prevent API error
  warnings.warn(


Bot: 
